# Step 3. Adaptation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/younesStrittmatter/sweetpea-starter/blob/main/tutorial/notebooks/3_adaptation.ipynb)

The tutorial step without an assistant. Run the cells from top to bottom; the tables below are from one run, yours will differ where the design leaves things open.

In [ ]:
%pip install -q sweetpea pandas

**So far.** Step 1 crossed the ink colour with congruency: half of the trials congruent. This step changes that share, and the share becomes the manipulation.

**The question.** Do people adjust to how frequent conflict is?

**Prediction.** The congruency effect is smaller in mostly incongruent blocks than in mostly congruent blocks.

**The alternative to rule out.** In a mostly congruent block the word predicts the ink, so people can simply learn to follow the word.

**The analysis.** The congruency effect in mostly congruent blocks against the congruency effect in mostly incongruent blocks.

### As in step 1

In [ ]:
import pandas as pd
from sweetpea import (Factor, DerivedLevel, WithinTrial, Transition, Exclude, MinimumTrials,
                      CrossBlock, synthesize_trials, CMSGen)

colours = ["red", "green", "blue"]

# regular factors: set freely on every trial
colour = Factor("colour", colours)      # the ink
word = Factor("word", colours)          # what the word says

In [ ]:
def same(colour, word):
    return colour == word


def different(colour, word):
    return colour != word

### New: a weight

A level can have a weight: the third argument of `DerivedLevel`. 3 to 1 makes the first level three times as frequent as the second. Crossed with the ink colour that is 12 combinations, two rounds: 18 of 24 trials.

In [ ]:
WEIGHTS = {"mostly congruent": (3, 1), "mostly incongruent": (1, 3)}     # (congruent, incongruent)


def block(context):
    congruent_weight, incongruent_weight = WEIGHTS[context]
    congruency = Factor("congruency", [
        DerivedLevel("congruent", WithinTrial(same, [colour, word]), congruent_weight),
        DerivedLevel("incongruent", WithinTrial(different, [colour, word]), incongruent_weight),
    ])
    return CrossBlock(design=[colour, word, congruency], crossing=[colour, congruency],
                      constraints=[MinimumTrials(24)])

### A session of four blocks

Each block is generated on its own and gets its context as a column.

In [ ]:
blocks = []
for number, context in enumerate(["mostly congruent", "mostly incongruent", "mostly incongruent", "mostly congruent"], start=1):
    trials = pd.DataFrame(synthesize_trials(block(context), 1, CMSGen)[0])
    trials.insert(0, "context", context)
    trials.insert(0, "block", number)
    blocks.append(trials)
session = pd.concat(blocks, ignore_index=True)
display(pd.crosstab(session["context"], session["congruency"]))

congruency,congruent,incongruent
context,,
mostly congruent,36,12
mostly incongruent,12,36


### How well does the word tell the ink?

For each word, the share of trials on which it is shown in its own colour. That number is the alternative explanation.

In [ ]:
session["word_is_ink"] = session["word"] == session["colour"]
display(session.groupby(["context", "word"])["word_is_ink"].mean().round(2).unstack())

word,blue,green,red
context,,,
mostly congruent,0.67,0.80,0.80
mostly incongruent,0.24,0.25,0.27


**Check.** Are the proportions what was asked for? How many incongruent trials does a mostly congruent block have: is that enough to measure the effect there? A word is in its own colour far more often in one context than in the other: how could learning and control be told apart?

**Try.** Change the weights to `(2, 1)` and `(1, 2)`. What are the proportions and the block lengths now?

**Next.** Step 4 asks whether this adjustment has a price when the task changes. There the weight meets a limit: it cannot depend on another factor.